[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juliopez/Taller-Fundamentos-Data-Science-Python/blob/main/Machine_Learning_2026/03_Notebooks/NB04_Introduccion_Practica_CNN.ipynb)

# NB04 — Introducción práctica a las Redes Neuronales Convolucionales (CNN)

**Curso:** Machine Learning 2026 · **Experiencia:** clasificación de imágenes de piedra, papel y tijera.

**Objetivo:** construir, entrenar y evaluar una CNN sencilla, reconociendo qué conceptos de las redes neuronales densas (MLP) se mantienen y qué operaciones nuevas permiten procesar imágenes.

**Ruta de trabajo:** cargar ZIP desde GitHub al computador → subirlos a Colab → verificar imágenes → preparar conjuntos → construir CNN → entrenar → evaluar y reflexionar.

> **Importante:** los notebooks anteriores de MLP trabajaron con **datos numéricos estructurados**, no con imágenes. En este notebook cambiamos **el tipo de dato** y **la arquitectura**; no estamos simplemente agregando una capa a la misma entrada numérica.

## 1. De una MLP con datos numéricos a una CNN con imágenes

En una **MLP** anterior, cada observación ingresaba como un **vector de variables numéricas** (por ejemplo, valores de indicadores). Utilizamos `Sequential`, capas `Dense`, activaciones, una función de pérdida, un optimizador y métricas.

Ahora cada observación será una **imagen RGB**. Una imagen de 128 × 128 píxeles tiene forma `(128, 128, 3)`: alto, ancho y tres canales de color. Aunque sus píxeles son números, su **posición espacial** importa.

Una MLP podría recibir una imagen previamente aplanada, pero perdería la organización espacial explícita de sus píxeles. Una **CNN** incorpora filtros convolucionales para aprender patrones locales antes de clasificar.

| Aspecto | MLP vista anteriormente | CNN de este notebook |
|---|---|---|
| Entrada | Variables numéricas tabulares | Tensor de píxeles RGB |
| Primeras capas | `Dense` | `Conv2D` y `MaxPooling2D` |
| Etapa final | `Dense` para clasificar | `Dense` para clasificar |
| Entrenamiento | `compile`, `fit`, `evaluate` | Los mismos procedimientos |

**Pregunta inicial:** ¿qué información conserva una imagen como matriz que no se observa directamente en un vector aplanado?

## 2. Preparar Google Colab

Trabajaremos en Google Colab. No se requiere montar Google Drive ni descargar el dataset automáticamente desde Internet.

**Antes de ejecutar la siguiente celda**, descarga **los tres ZIP** desde el [directorio del dataset en GitHub](https://github.com/juliopez/Taller-Fundamentos-Data-Science-Python/tree/main/Machine_Learning_2026/02_Datasets/dataset_cnn):

- `dataset_cnn_rock.zip`
- `dataset_cnn_paper.zip`
- `dataset_cnn_scissors.zip`

En la página de cada archivo selecciona la opción **Download raw file**. Después, en Colab, elige los tres ZIP cuando aparezca el selector.

**Nota:** el almacenamiento de Colab es temporal; al reiniciar la sesión puede ser necesario volver a cargar los archivos.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile

archivos_esperados = {
    "dataset_cnn_rock.zip",
    "dataset_cnn_paper.zip",
    "dataset_cnn_scissors.zip",
}
print("Selecciona los tres ZIP descargados desde GitHub.")
archivos_subidos = files.upload()
faltantes = archivos_esperados - set(archivos_subidos)
if faltantes:
    raise FileNotFoundError(f"Faltan ZIP: {sorted(faltantes)}")

directorio_base = Path("/content")
for nombre in sorted(archivos_esperados):
    with ZipFile(nombre) as comprimido:
        # El dataset es material docente conocido: comprobar rutas antes de extraer.
        for entrada in comprimido.infolist():
            ruta = Path(entrada.filename)
            if ruta.is_absolute() or ".." in ruta.parts or ruta.parts[0] != "dataset_cnn":
                raise ValueError(f"Ruta no válida en {nombre}: {entrada.filename}")
        comprimido.extractall(directorio_base)

DATASET_DIR = directorio_base / "dataset_cnn"
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"
print("Dataset disponible en:", DATASET_DIR)

## 3. Inspeccionar el dataset antes de entrenar

Las carpetas determinan las **etiquetas**: `rock`, `paper` y `scissors`. Conservamos la separación original entre **entrenamiento** y **prueba**. Más adelante reservaremos una parte del entrenamiento para **validación**.

**Comprobaciones:** existencia de carpetas, cantidad de imágenes por categoría y ejemplos visuales. Esta inspección no sustituye una auditoría exhaustiva de calidad.

In [ ]:
CLASES = ["paper", "rock", "scissors"]  # Orden alfabético, usado de forma consistente
for particion, carpeta, esperado in [("train", TRAIN_DIR, 150), ("test", TEST_DIR, 50)]:
    for clase in CLASES:
        ruta = carpeta / clase
        imagenes = sorted(ruta.glob("*.png"))
        print(f"{particion:5} / {clase:8}: {len(imagenes):3} imágenes")
        if len(imagenes) != esperado:
            raise ValueError(f"Cantidad inesperada en {ruta}: {len(imagenes)}")
print("Estructura y recuentos correctos.")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, ejes = plt.subplots(1, 3, figsize=(10, 3))
for eje, clase in zip(ejes, CLASES):
    ejemplo = sorted((TRAIN_DIR / clase).glob("*.png"))[0]
    with Image.open(ejemplo) as imagen:
        eje.imshow(imagen.convert("RGB"))
    eje.set_title(clase)
    eje.axis("off")
plt.suptitle("Ejemplos del dataset de entrenamiento")
plt.tight_layout()
plt.show()

## 4. Preparar los conjuntos de entrenamiento, validación y prueba

Usaremos **128 × 128 píxeles**, un tamaño suficiente para esta primera experiencia y más económico que trabajar con imágenes originales grandes.

- **Entrenamiento:** 80 % de las 450 imágenes de `train/` = **360 imágenes**.
- **Validación:** 20 % de las 450 imágenes de `train/` = **90 imágenes**.
- **Prueba:** las **150 imágenes** originales de `test/`, sin mezclarlas con entrenamiento ni validación.

La división será **estratificada**: conserva 120 imágenes de entrenamiento y 30 de validación por categoría. Utilizaremos una semilla fija para reproducibilidad.

La **normalización** transforma los valores de píxeles de 0–255 a 0–1. Este paso facilita el entrenamiento, pero no modifica la categoría de la imagen.

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

SEED = 42
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

rutas_train = []
etiquetas_train = []
rutas_test = []
etiquetas_test = []

for indice, clase in enumerate(CLASES):
    for ruta in sorted((TRAIN_DIR / clase).glob("*.png")):
        rutas_train.append(str(ruta))
        etiquetas_train.append(indice)
    for ruta in sorted((TEST_DIR / clase).glob("*.png")):
        rutas_test.append(str(ruta))
        etiquetas_test.append(indice)

x_train, x_val, y_train, y_val = train_test_split(
    rutas_train, etiquetas_train,
    test_size=0.20, random_state=SEED, stratify=etiquetas_train
)
print("Entrenamiento:", len(x_train), "Validación:", len(x_val), "Prueba:", len(rutas_test))
assert (len(x_train), len(x_val), len(rutas_test)) == (360, 90, 150)

### 4.1. Convertir archivos PNG en tensores

La función siguiente lee cada imagen, la convierte a **RGB**, la redimensiona y la normaliza. `tf.data.Dataset` permite procesar imágenes por **lotes** sin cargar todas simultáneamente en memoria.

El valor `batch_size=16` significa que la red recibe grupos de 16 imágenes en cada paso de entrenamiento.

In [ ]:
def leer_imagen(ruta, etiqueta):
    contenido = tf.io.read_file(ruta)
    imagen = tf.io.decode_png(contenido, channels=3)
    imagen = tf.image.resize(imagen, IMG_SIZE)
    imagen = tf.cast(imagen, tf.float32) / 255.0
    return imagen, etiqueta

def construir_dataset(rutas, etiquetas, entrenamiento=False):
    ds = tf.data.Dataset.from_tensor_slices((
        tf.constant(rutas), tf.constant(etiquetas, dtype=tf.int32)
    ))
    if entrenamiento:
        ds = ds.shuffle(buffer_size=len(rutas), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(leer_imagen, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = construir_dataset(x_train, y_train, entrenamiento=True)
val_ds = construir_dataset(x_val, y_val)
test_ds = construir_dataset(rutas_test, etiquetas_test)

imagenes_lote, etiquetas_lote = next(iter(train_ds))
print("Forma de un lote de imágenes:", imagenes_lote.shape)
print("Forma de las etiquetas:", etiquetas_lote.shape)
print("Rango de píxeles:", float(tf.reduce_min(imagenes_lote)), float(tf.reduce_max(imagenes_lote)))

## 5. ¿Qué hace cada capa de una CNN?

Construiremos una arquitectura **deliberadamente pequeña** para observar cómo se transforma una imagen. Los conceptos nuevos son las capas convolucionales y de reducción espacial; el final de la red utiliza las capas densas ya conocidas.

### 5.1. `Input`: forma de la imagen

`Input(shape=(128, 128, 3))` indica que cada ejemplo contiene **128 filas, 128 columnas y 3 canales RGB**. No es una capa que aprenda patrones: declara la forma esperada de los datos.

### 5.2. `Conv2D`: aprender filtros visuales

`Conv2D(16, (3, 3), activation="relu")` aprende **16 filtros** de tamaño **3 × 3**. Cada filtro recorre regiones pequeñas de la imagen y produce un **mapa de características**. Durante el entrenamiento, la red ajusta los valores de esos filtros para detectar patrones útiles.

Con el `padding="valid"` predeterminado y paso 1, la salida de la primera convolución pasa de **128 × 128 × 3** a **126 × 126 × 16**.

### 5.3. `ReLU`: incorporar no linealidad

La activación `relu` transforma valores negativos en cero y conserva los positivos. Esto permite aprender relaciones no lineales. Ya conocíamos ReLU de las MLP; aquí la aplicamos a los **mapas de características**.

### 5.4. `MaxPooling2D`: reducir las dimensiones espaciales

`MaxPooling2D((2, 2))` examina regiones de 2 × 2 y conserva el **valor máximo** de cada región. Reduce aproximadamente a la mitad el alto y ancho del mapa, sin modificar el número de canales.

**Ejemplo:**

```text
Entrada (2×2):       Salida:
1  4
3  2                  4
```

Después de la primera convolución, el mapa de **126 × 126 × 16** pasa a **63 × 63 × 16**.

**Pregunta:** ¿qué información podríamos perder al conservar solamente los máximos?

### 5.5. Segunda convolución y segundo pooling

Una segunda `Conv2D` con **32 filtros** procesa los patrones detectados por la etapa anterior. A continuación, otro `MaxPooling2D` reduce las dimensiones:

```text
63 × 63 × 16 → Conv2D(32, 3×3) → 61 × 61 × 32
61 × 61 × 32 → MaxPooling2D(2×2) → 30 × 30 × 32
```

**Interpretación:** disminuyen las dimensiones espaciales, pero aumenta el número de mapas de características.

### 5.6. `Flatten`: conectar las características con la MLP

`Flatten()` transforma los mapas de **30 × 30 × 32** en **un vector de 28.800 valores**. No aprende parámetros: reorganiza los valores para entregarlos a una capa densa.

Aquí aparece el **puente con las MLP anteriores**: desde este punto podemos utilizar capas `Dense`, como ya hicimos con datos numéricos.

### 5.7. `Dense` y `Softmax`: clasificar

- `Dense(32, activation="relu")` combina las características visuales extraídas.
- `Dense(3, activation="softmax")` produce **tres probabilidades**, una por clase, cuya suma es aproximadamente 1.

El orden de las clases será: **0 = paper, 1 = rock, 2 = scissors**.

**Importante:** la red no reconoce automáticamente el significado de estas palabras; aprende asociaciones entre imágenes y etiquetas numéricas.

## 6. Construir nuestra primera CNN

Observa que `Sequential`, `Dense` y las activaciones son conceptos familiares de los ejercicios de MLP. Las novedades son `Conv2D`, `MaxPooling2D` y `Flatten`.

Esta CNN es intencionalmente básica: no incorpora transferencia de aprendizaje, fine-tuning ni técnicas avanzadas de optimización. Esos contenidos corresponden a experiencias posteriores.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

modelo_cnn = keras.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Conv2D(16, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(32, activation="relu"),
    layers.Dense(3, activation="softmax"),
], name="cnn_piedra_papel_tijera")

modelo_cnn.summary()

## 7. Interpretar `model.summary()`

La salida de `summary()` permite responder:

1. ¿Cuántos mapas de características genera cada `Conv2D`?
2. ¿Cómo cambian alto y ancho después de `MaxPooling2D`?
3. ¿Por qué `Flatten` genera **28.800 valores**?
4. ¿Cuántos parámetros entrenables tiene la primera capa `Dense`?

**Atención:** `Flatten` no reduce parámetros. Al conectar un vector grande con una capa densa, el número de pesos puede crecer considerablemente. Más adelante estudiaremos alternativas más eficientes.

### 7.1. Comprobar las dimensiones programáticamente

In [ ]:
for capa in modelo_cnn.layers:
    print(f"{capa.name:25} → {capa.output.shape}")

## 8. Compilar la CNN: reutilizamos lo aprendido con MLP

El procedimiento es el mismo que conocimos antes:

- **Optimizador `adam`:** ajusta los pesos durante el entrenamiento.
- **Pérdida `sparse_categorical_crossentropy`:** apropiada para tres clases mutuamente excluyentes cuyas etiquetas son números enteros (`0, 1, 2`).
- **Métrica `accuracy`:** proporción de predicciones correctas.

La compilación **no agrega capas**: define cómo se optimizará y evaluará el aprendizaje.

In [ ]:
modelo_cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## 9. Entrenar la CNN

Entrenaremos inicialmente durante **5 épocas**, suficientes para observar el procedimiento sin buscar una precisión óptima. Cada época recorre el conjunto de entrenamiento completo.

El conjunto de **validación** permite observar cómo se comporta la red con imágenes que no utiliza para ajustar sus pesos. El conjunto de **prueba** se mantiene reservado hasta la evaluación final.

> Los tiempos y resultados pueden variar según la disponibilidad de CPU/GPU y las condiciones de la sesión de Colab.

In [ ]:
EPOCAS = 5

historial = modelo_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCAS,
    verbose=1,
)

## 10. Interpretar las curvas de aprendizaje

Analizaremos dos métricas:

- **Accuracy:** porcentaje de clasificaciones correctas; valores más altos suelen ser mejores.
- **Loss:** error utilizado para ajustar los pesos; valores más bajos suelen ser mejores.

No basta con observar la exactitud del entrenamiento. Una brecha grande entre entrenamiento y validación puede sugerir **sobreajuste**, aunque con pocos datos y épocas no siempre es concluyente.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(historial.history["accuracy"], marker="o", label="Entrenamiento")
ejes[0].plot(historial.history["val_accuracy"], marker="o", label="Validación")
ejes[0].set_title("Accuracy")
ejes[0].set_xlabel("Época")
ejes[0].legend()
ejes[0].grid(alpha=0.3)

ejes[1].plot(historial.history["loss"], marker="o", label="Entrenamiento")
ejes[1].plot(historial.history["val_loss"], marker="o", label="Validación")
ejes[1].set_title("Loss")
ejes[1].set_xlabel("Época")
ejes[1].legend()
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Evaluar con imágenes de prueba

Las **150 imágenes de prueba** se mantuvieron fuera del entrenamiento y de la validación. Las utilizaremos ahora para obtener una primera estimación del rendimiento con ejemplos no utilizados para ajustar el modelo.

Una buena exactitud en este dataset pequeño **no garantiza** el mismo resultado en fotografías tomadas con otras cámaras, fondos o condiciones de iluminación.

In [ ]:
perdida_test, exactitud_test = modelo_cnn.evaluate(test_ds, verbose=0)
print(f"Loss de prueba: {perdida_test:.4f}")
print(f"Accuracy de prueba: {exactitud_test:.2%}")

### 11.1. Visualizar predicciones de las tres categorías

La red devuelve tres probabilidades por imagen. `argmax` selecciona la categoría de mayor probabilidad.

**¿Por qué modificamos esta visualización?** El dataset de prueba está organizado por carpetas y clases. Tomar solamente el primer lote puede mostrar imágenes de una única categoría y ofrecer una visión incompleta. Por eso seleccionaremos **cuatro imágenes aleatorias por clase**, usando una semilla fija, para comparar aciertos y errores de `paper`, `rock` y `scissors`.

La visualización representa **una muestra de 12 imágenes**; no reemplaza la evaluación completa de las 150 imágenes.

In [ ]:
# Muestra reproducible y equilibrada: cuatro imágenes de cada categoría.
rng = np.random.default_rng(SEED)
rutas_muestra, etiquetas_muestra = [], []
for indice, clase in enumerate(CLASES):
    candidatos = sorted((TEST_DIR / clase).glob("*.png"))
    seleccion = rng.choice(len(candidatos), size=4, replace=False)
    for posicion in seleccion:
        rutas_muestra.append(str(candidatos[int(posicion)]))
        etiquetas_muestra.append(indice)

muestra_ds = construir_dataset(rutas_muestra, etiquetas_muestra)
imagenes_muestra, etiquetas_reales = next(iter(muestra_ds))
probabilidades = modelo_cnn.predict(imagenes_muestra, verbose=0)
predicciones = np.argmax(probabilidades, axis=1)

fig, ejes = plt.subplots(3, 4, figsize=(12, 9))
for i, eje in enumerate(ejes.flat):
    eje.imshow(imagenes_muestra[i].numpy())
    real = CLASES[int(etiquetas_reales[i])]
    predicha = CLASES[int(predicciones[i])]
    confianza = float(np.max(probabilidades[i]))
    eje.set_title(f"Real: {real}\nPred.: {predicha} ({confianza:.0%})",
                   color="green" if real == predicha else "red")
    eje.axis("off")
plt.suptitle("Cuatro ejemplos por categoría — verde: acierto; rojo: error")
plt.tight_layout()
plt.show()

### 11.2. Matriz de confusión: ¿qué clases confunde la CNN?

La **matriz de confusión** compara la clase real con la clase predicha **para las 150 imágenes de prueba**.

- **Filas:** categoría real.
- **Columnas:** categoría predicha.
- **Diagonal principal:** clasificaciones correctas.
- **Fuera de la diagonal:** errores; permiten identificar entre qué categorías se confunde el modelo.

Por ejemplo, un valor en la fila `paper` y columna `scissors` indica cuántas imágenes de papel fueron clasificadas como tijeras.

**Pregunta de análisis:** ¿qué categoría tiene más errores? ¿El 76 % de accuracy de prueba se distribuye de manera similar entre las tres clases?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Evaluación de TODAS las imágenes de prueba, sin mezclar con entrenamiento.
y_reales = np.array(etiquetas_test, dtype=int)
y_probabilidades = modelo_cnn.predict(test_ds, verbose=0)
y_predichas = np.argmax(y_probabilidades, axis=1)

matriz = confusion_matrix(y_reales, y_predichas, labels=range(len(CLASES)))
fig, eje = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=matriz, display_labels=CLASES).plot(
    ax=eje, cmap="Blues", values_format="d", colorbar=False
)
eje.set_title("Matriz de confusión — conjunto de prueba")
plt.tight_layout()
plt.show()

print("Informe por categoría:")
print(classification_report(y_reales, y_predichas,
                            labels=range(len(CLASES)),
                            target_names=CLASES, zero_division=0))

## 12. Reflexión final: ¿qué aprendimos?

Responde brevemente:

1. En las MLP anteriores trabajamos con variables numéricas estructuradas. ¿Qué cambió al utilizar imágenes?
2. ¿Cuál es la función de `Conv2D` y qué representan sus filtros?
3. ¿Qué información conserva y qué información reduce `MaxPooling2D`?
4. ¿Por qué necesitamos `Flatten` antes de la primera `Dense` de esta arquitectura?
5. ¿Qué elementos del entrenamiento son iguales a los utilizados con MLP?
6. ¿Qué diferencias observaste entre accuracy de entrenamiento, validación y prueba?
7. Según la matriz de confusión, ¿qué categorías se confunden más? ¿Qué hipótesis planteas para explicar los errores?

### Lista de verificación

- [ ] Cargué los **tres ZIP** desde mi computador a Colab.
- [ ] Verifiqué **450 imágenes originales de entrenamiento** y **150 de prueba**.
- [ ] Separé **360 imágenes para entrenamiento** y **90 para validación**.
- [ ] Comprendí las funciones de `Conv2D`, `MaxPooling2D` y `Flatten`.
- [ ] Construí, compilé y entrené una CNN.
- [ ] Interpreté curvas de aprendizaje, predicciones equilibradas por clase y matriz de confusión.

**Continuidad:** este notebook introduce una primera CNN funcional. En el **NB05** profundizaremos en la construcción y entrenamiento de redes convolucionales.

## 13. Actividad autónoma de experimentación: ¿podemos mejorar nuestra CNN?

### 13.1. Propósito

Ya construimos y evaluamos nuestra primera CNN. Ahora investigaremos **cómo una modificación controlada puede cambiar su comportamiento**. No se trata de obtener obligatoriamente un accuracy mayor, sino de **formular una hipótesis, experimentar y justificar una conclusión con evidencia**.

**Pregunta orientadora:** ¿qué componente modificarías para intentar mejorar la capacidad de generalización de la CNN y por qué?

> **Regla metodológica:** modifica **una sola variable por experimento** y conserva las demás condiciones (dataset, particiones, semilla, arquitectura restante y procedimiento de evaluación). Al cambiar la arquitectura o las épocas, crea y entrena **un modelo nuevo desde cero**, no continúes entrenando el modelo anterior. De esta manera, la comparación será más interpretable.

### 13.2. Elige un experimento

Selecciona **uno** de los siguientes tres experimentos. El resto de la clase podrá trabajar con alternativas diferentes para compartir los resultados.

| Experimento | Configuración original | Modificación propuesta | Pregunta de investigación |
|---|---|---|---|
| **A. Épocas** | 5 épocas | 10 épocas | ¿Entrenar más mejora la validación o aumenta el sobreajuste? |
| **B. Filtros convolucionales** | `Conv2D(16)` y `Conv2D(32)` | `Conv2D(32)` y `Conv2D(64)` | ¿Aprender más mapas de características mejora el rendimiento? |
| **C. Neuronas densas** | `Dense(32)` | `Dense(64)` | ¿Aumentar la capacidad de clasificación mejora la validación? |

**Orientación:** los cambios B y C aumentan los parámetros del modelo y pueden incrementar el tiempo de entrenamiento; **no garantizan** mejores predicciones. El cambio A puede mejorar el aprendizaje, pero también favorecer el sobreajuste.

### 13.3. Antes de modificar: formula tu hipótesis

Completa estas frases **antes** de ejecutar el experimento:

- **Componente elegido:** …
- **Modificación exacta:** …
- **Hipótesis:** «Si modifico …, entonces espero que …, porque …».
- **Indicadores que observaré:** accuracy de validación, loss de validación, diferencia entre entrenamiento y validación, y tiempo de entrenamiento.

### 13.4. Realiza el experimento

1. Conserva los resultados del **modelo original de cinco épocas** como referencia.
2. Modifica **solo** el componente seleccionado en el código de construcción o entrenamiento.
3. Inicializa y compila una **nueva CNN**, utilizando la misma semilla, las mismas imágenes y la misma separación entrenamiento/validación.
4. Entrena el nuevo modelo y observa las curvas de `accuracy` y `loss`.
5. Compara los resultados de validación y registra el tiempo de entrenamiento.

**Importante sobre la evaluación:** utiliza **validación** para comparar configuraciones y decidir qué modelo conservar. **No utilices repetidamente las 150 imágenes de prueba para elegir la mejor arquitectura**: hacerlo convertiría el test en parte del proceso de ajuste. Cuando hayas seleccionado una configuración, evalúala **una sola vez** sobre el conjunto de prueba y comunica ese resultado por separado.

### 13.5. Registra y compara los resultados

| Modelo | Variable modificada | Accuracy validación | Loss validación | Tiempo entrenamiento |
|---|---|---|---|---|
| CNN original | Ninguna (5 épocas; 16/32 filtros; 32 neuronas) | Completar | Completar | Completar |
| CNN experimental | Completar | Completar | Completar | Completar |

**Nota:** registra las métricas de la **última época** de cada entrenamiento y, cuando sea pertinente, observa también la evolución de las curvas. Las comparaciones con un único entrenamiento son **exploratorias**: existe variabilidad incluso al fijar la semilla.

### 13.6. Analiza y fundamenta

Responde brevemente:

1. ¿Qué modificación realizaste y por qué esperabas que fuera útil?
2. ¿Se cumplió tu hipótesis? Fundamenta con los resultados de **validación**.
3. ¿Cómo cambiaron las curvas de entrenamiento y validación? ¿Observas indicios de sobreajuste?
4. ¿El tiempo de entrenamiento cambió? ¿Consideras que el costo adicional se justifica?
5. ¿Conservarías la CNN original o la experimental? ¿Por qué?
6. Tras seleccionar tu modelo, ¿cómo interpretarías una diferencia entre su resultado de validación y el resultado final de prueba?

### 13.7. Cierre de la actividad

Comparte tu experimento con el curso. Si otros estudiantes modificaron componentes diferentes, comparen sus **hipótesis y resultados**, recordando que se trata de pruebas exploratorias.

**Conclusión clave:** en Machine Learning, modificar una red no equivale automáticamente a mejorarla. Una decisión técnica debe justificarse con evidencia y con una evaluación que respete la separación entre entrenamiento, validación y prueba.

> **Próxima experiencia:** profundizaremos en estrategias para construir y entrenar redes convolucionales.